# DamageNet-VGG16 — demo walkthrough
Transfer learning with VGG16 for road-damage patch classification on RDD2022 (SRM IST 21CSC406J case study).

This notebook only **loads saved results and figures** — run `python -m src.run_all` first.

In [ ]:
import json, sys
from pathlib import Path
from IPython.display import Image, Markdown, display
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
R = ROOT / 'results'; F = ROOT / 'figures'

## 1. Results table

In [ ]:
display(Markdown((R / 'summary.md').read_text()))

## 2. Data: label distribution and sample patches

In [ ]:
display(Image(F / 'label_distribution.png', width=900))
display(Image(F / 'sample_patches.png', width=700))

## 3. Model comparison and regularisation ablation

In [ ]:
for n in ['model_comparison', 'ablation', 'learning_curves']:
    display(Image(F / f'{n}.png', width=800))

## 4. Fine-tuned VGG16: confusion matrix and per-class report

In [ ]:
ft = json.loads((R / 'vgg16_ft.json').read_text())
print(f"test accuracy {ft['test_accuracy']:.4f}, macro F1 {ft['macro_f1']:.4f}")
for c, v in ft['classification_report'].items():
    if isinstance(v, dict):
        print(f"{c:>12}  P {v['precision']:.3f}  R {v['recall']:.3f}  F1 {v['f1-score']:.3f}")
display(Image(F / 'confusion_matrix.png', width=520))

## 5. Error analysis

In [ ]:
an = json.loads((R / 'analysis.json').read_text())
print('crack-type share of errors:', an['crack_confusion_share'])
print('top confusions:', [(t['true'], t['pred'], t['count']) for t in an['top_confusions']])
print('by country:', {k: v['accuracy'] for k, v in an['accuracy_by_country'].items()})
print('by box size:', {k: v['accuracy'] for k, v in an['accuracy_by_box_size'].items()})
display(Image(F / 'misclassified.png', width=850))

## 6. Predict one test patch with the saved checkpoint (optional)
Needs `checkpoints/vgg16_ft.pt` and the patch dataset.

In [ ]:
import numpy as np, torch
from src.models import VGG16Classifier
ck = ROOT / 'checkpoints' / 'vgg16_ft.pt'
npz = ROOT / 'data' / 'patches' / 'test.npz'
if ck.exists() and npz.exists():
    c = torch.load(ck, map_location='cpu')
    m = VGG16Classifier(c['hidden'], c['dropout'], 4, pretrained=False); m.load_state_dict(c['state_dict']); m.eval()
    d = np.load(npz); i = 7
    x = torch.from_numpy(d['x'][i]).permute(2, 0, 1).float().div(255)[None]
    p = torch.softmax(m(x), 1)[0].detach().numpy()
    print('true', c['classes'][d['y'][i]], '| pred', c['classes'][p.argmax()], np.round(p, 3))
else:
    print('checkpoint or patches not available — skipped')